# 02 · lumen against Geant4 and WCSim

Is lumen's light the light Geant4 makes? We check it three ways on Super-K, then on WCTE:

1. **The same event.** Notebook 00's muon and electron: Geant4 tracked their light itself and recorded every photon
   a PMT absorbed (a *hit*). We send the same Geant4 steps through lumen and compare, PMT by PMT, the number of
   hits and when they arrive.
2. **Many events.** A hundred muons and a hundred electrons at random places and directions, each through Geant4
   and lumen: whether lumen is right on average and in its spread, sensor by sensor and event by event.
3. **The family's charge check.** Plain Geant4 10.7.4 (`g4ref`) and WCSim (the experiments' Geant4 application for
   water Cherenkov detectors) fired flashes of light of one colour from a few points; `lumen.compare.charge` fires
   the same flashes in lumen and compares every PMT. These reference runs are listed, with their provenance, in
   lumen-wc's manifest (`lumen_wc.REFERENCES`).

Reading the numbers: a **ratio** of totals lumen / reference with its error and **z** = (ratio - 1) / error
(|z| <= 3 passes); a **chi2/ndf** over PMTs, about 1 when the PMTs differ only by chance.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown

import lumen
import lumen_wc
from lumen.compare import charge, event, references, report
from lumen.tools import plot

WORK = Path(os.environ.get("LUMEN_NOTEBOOKS", "lumen-notebooks"))
WORK.mkdir(parents=True, exist_ok=True)


def detector(name):
    """Load a lumen-wc detector, converting it the first time (notebook 01)."""
    path = WORK / f"{name}.lumen"
    if not path.exists():
        lumen.convert(f"lumen_wc:{name}", out=path, data=references.store())
    return path, lumen.load(path)


sk_path, sk = detector("sk")
events = references.product("g4ref-events-wc-933e631d", extra=[lumen_wc.REFERENCES])
gun = references.product("g4ref-gun-sk-9fd555c0", extra=[lumen_wc.REFERENCES])  # the many events (section 2)

## 1. The same event, through lumen and through Geant4

`event.compare` runs the steps through lumen and returns, per PMT, lumen's expected hits and Geant4's hits, and the
arrival times of every hit. lumen counts what Geant4 counts: a photon absorbed at the PMT's photocathode (before
the quantum efficiency). One event has tens of thousands of hits, so each PMT's count fluctuates by its square root:
the pulls are about a normal distribution of width 1, with a longer tail to the right where a PMT has few hits
(Poisson counts are skewed); section 2 compares them with what an exact match gives.

In [ ]:
results = {}
for name in ("sk_mu600", "sk_e600"):
    ev = event.read(events / f"{name}.npz")
    res = event.compare(sk, ev, samples=2**18, seed=0)
    results[name] = (ev, res)
    s = res["summary"]
    print(f"{name}: lumen / Geant4 = {s['ratio']:.4f} ± {s['err']:.4f} (z {s['z']:+.1f}); "
          f"chi2/ndf {s['chi2'] / s['ndf']:.2f} over {s['ndf']} PMTs; mean time lumen - Geant4 {s['dt']:+.2f} ns")
    plot.compare(res, title=name)
plt.show()

The muon side by side: Geant4's hits (one event, so grainy) and lumen's expectation.

In [ ]:
ev, res = results["sk_mu600"]
top = max(res["geant4"].max(), res["lumen"].max())
plot.display(sk, res["geant4"], title="Geant4 hits (one event)", label="hits", log=True, vmax=top)
plot.display(sk, res["lumen"], title="lumen (expected hits)", label="hits", log=True, vmax=top)
plt.show()

## 2. Many events

One event shows lumen and Geant4 agree on that event; many tell whether lumen is right on average and in its
spread. The product `g4ref-gun-sk-9fd555c0` holds 100 muons and 100 electrons of 600 MeV, each from a random place
2 m inside the PMTs, in a random direction, through plain Geant4 10.7.4 (`g4ref`, WCSim's hit rule, as the charge
check's references). `event.compare_events` runs each through lumen twice with independent seeds: one run chooses
which PMTs enter a statistic and sets its variance, the other is compared with Geant4, so lumen's own noise cannot
bias the result. `N_EVENTS` sets how many of each particle (more: slower, and tighter).

Reading the figure, left to right:

- **per event**: Geant4's total hits against lumen's, and each event's pull, (Geant4 - lumen) / sigma, against N(0, 1);
- **per PMT and event**: the pulls of every PMT with at least 5 expected hits, against the same statistic of
  Poisson counts about lumen (an exact match: skewed to the right where counts are few, so not quite N(0, 1));
- **by expected hits**: Geant4 against lumen summed over PMTs in bins of how much light lumen expects, in percent:
  a difference between bright and dim PMTs (light moved from one to the other) shows here even when totals agree;
- **arrival times**: every hit of every event, and their ratio in bins of equal light (the late tail is the
  muon's decay electron, at microseconds).

Then each PMT over all the events, on the detector: a region where lumen and Geant4 differ shows as a patch of one
colour.

In [ ]:
N_EVENTS = 20  # events per particle, up to 100

many = {}
for particle in ("mu600", "e600"):
    files = sorted(gun.glob(f"sk_{particle}_*.npz"))[:N_EVENTS]
    many[particle] = event.compare_events(sk, files, samples=2**18, seed=0, dt=4.0, nt=1000)
    r = many[particle]
    print(f"{particle}: {len(files)} events, Geant4 / lumen {r['geant4'].sum() / r['lumen'].sum():.4f}; "
          f"per event sd {r['event_z'].std(ddof=1):.2f}; per PMT pull mean {r['pull'].mean():+.3f} "
          f"(exact match {r['pull_toy'].mean():+.3f}), sd {r['pull'].std():.3f} ({r['pull_toy'].std():.3f})")
    for q in r["bins"]:
        print(f"   expected {q['lo']:>6g}-{q['hi']:<6g} hits: Geant4 - lumen {100 * q['diff']:+.2f} ± {100 * q['err']:.2f} %")
    plot.compare_events(r, title=f"Super-K, {particle}: {len(files)} Geant4 events")
    plt.show()

In [ ]:
plot.display(sk, many["mu600"]["sensor_z"], diverging=True, label="z over the events", title="muons: each PMT")
plt.show()

## 3. The family's charge check: Super-K against Geant4 and WCSim

Five flashes in Super-K, from two references kept in the reference store. Where they differ lumen follows plain
Geant4, so the Geant4 entry (`sk_g4`) passes or fails and WCSim's (`sk`) is reported beside it as a cross-check.
Geant4 ran lumen's **overlap-free export** of the detector (`lumen.compare.ownership`): where the GDML's volumes
overlap, Geant4's choice of owner is undefined, so it runs a GDML in which every shared region has lumen's owner.
`photons_factor` sets lumen's photons as a share of the reference's; we use a tenth to keep this quick, so lumen's
own noise is part of the errors.

In [ ]:
rows = []
for name in ("sk_g4", "sk"):
    spec = references.spec("charge", name, [lumen_wc.REFERENCES])
    gated = references.gated("charge", spec, [lumen_wc.REFERENCES])
    print(f"{name}: reference {spec['reference']} ({'passes or fails' if gated else 'reported'})")
    rows += charge.check(name, detector=sk_path, extra=[lumen_wc.REFERENCES], photons_factor=0.1)
Markdown(report.table(rows))

## WCTE against the same two references

The Water Cherenkov Test Experiment has the same two: WCSim (the experiment's software, on its own Geant4 build)
and plain Geant4 10.7.4 (`g4ref`) on lumen's overlap-free export of the detector. Again the plain Geant4 entry
(`wcte_g4`) passes or fails and WCSim's is reported beside it.

In [ ]:
wcte_path, wcte = detector("wcte")
rows = []
for name in ("wcte", "wcte_g4"):
    spec = references.spec("charge", name, [lumen_wc.REFERENCES])
    gated = references.gated("charge", spec, [lumen_wc.REFERENCES])
    print(f"{name}: reference {spec['reference']} ({'passes or fails' if gated else 'reported'})")
    rows += charge.check(name, detector=wcte_path, extra=[lumen_wc.REFERENCES], photons_factor=0.1)
Markdown(report.table([r for r in rows if r["quantity"] in ("ratio", "truncated weight")]))

**Next:** 03 explains derivatives of a simulation: what they are and how to trust them.